<div style="background:linear-gradient(to right,#98929F,#D3ACB5,#D3C7D3,#EFEAF0); padding:26px 34px; border-radius:10px;">
  <span style="font-size:26pt; color:#ffffff;"><b>Final Project &mdash; ECON&nbsp;320</b></span><br>
  <span style="font-size:14pt; color:#ffffff;">Introduction to Econometrics &nbsp;&middot;&nbsp; Multiple Regression &amp; Inference</span>
</div>
<hr style="border-top:6px solid #0C5A5C; margin:16px 0;">

### Flight Distance and Arrival Delays in 2024

**Team:** Group 7 — Lauren Patterson, William Sun, Jenny Huang, Chetan Desai &nbsp;&nbsp;&middot;&nbsp;&nbsp; **Section 2** &nbsp;&nbsp;&middot;&nbsp;&nbsp; **Date:** November 25, 2026

<div style="border-left:6px solid #D3ACB5; background:#faf6f8; padding:12px 18px; border-radius:0 6px 6px 0;">
<b>How to use this template.</b> This is a <b>starting point, not a checklist.</b> You already have most of the raw material from <b>Building Block 1</b> (your question, ranked ideas, data source, mini-codebook) and <b>Building Block 2</b> (your sign predictions, planned graphs, and first look at the data). Paste that work in and build on it.
<br><br>
<b>Make it yours:</b> rename sections, add or delete cells, and change the styling freely. The only hard rules are the ones on the Canvas project page: the document must read like a research paper <b>even with the code hidden</b>, your code must be commented, and you submit both the <b>HTML</b> export and the <code>.ipynb</code>.
</div>

## Contents

0. [Setup &mdash; packages and your data](#setup)
1. [Introduction and motivation](#intro)
2. [Data](#data)
3. [Descriptive statistics and correlations](#descriptives)
4. [Charts and plots](#graphs)
5. [Methodology / empirical model](#methodology)
6. [Results and robustness](#results)
7. [Conclusion](#conclusion)
8. [References](#references)



<a id="setup"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>0 &nbsp;Setup &mdash; packages and your data</b></span></div>

In [ ]:
# Packages used throughout the project. Run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Display settings (optional, just makes output easier to read)
pd.options.display.float_format = '{:.3f}'.format   # 3 decimals, no sci notation
sns.set_theme(style='whitegrid')                     # clean plot style
import statsmodels.api as sm
from statsmodels.stats import diagnostic as sm_diagnostic
from IPython.display import display
# If needed, run %pip install pyarrow in a separate cell.

**Load your data.** Put your data file in the **same folder as this notebook**, then set its name below. The cell checks the file is there and loads it into a DataFrame called `df`.

In [ ]:
import os
DATA_FILE = 'flights_2024_combined.parquet'
cols = ['Year','Month','DayOfWeek','ArrDelayMinutes','Distance','Reporting_Airline','Cancelled','Diverted','Origin']
raw = pd.read_parquet(DATA_FILE, columns=cols)
print('Original rows:', len(raw))
print(raw.groupby(['Year','Month']).size())
# Keep completed, non-diverted flights with complete analysis fields.
eligible = raw.loc[(raw.Year == 2024) & (raw.Cancelled == 0) & (raw.Diverted == 0)].dropna(subset=cols)
print('Eligible rows:', len(eligible))
df = eligible.sample(n=min(20000,len(eligible)), random_state=42).copy()
df['distance100'] = df.Distance / 100
df['Friday'] = (df.DayOfWeek == 5).astype(int)
df['December'] = (df.Month == 12).astype(int)
print('Analysis rows:',len(df), 'Months:',sorted(df.Month.unique()))
df.head()

<a id="intro"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>1 &nbsp;Introduction and motivation</b></span></div>

We examine whether longer domestic flights are associated with greater arrival delays in 2024, controlling for airline and calendar differences. Delays impose time costs on passengers and complicate airline scheduling. Our earlier proposal found a near-zero raw correlation between distance and delay; we now examine whether that relationship persists after accounting for observable differences. We focus on distance because it is known before departure, unlike attributed delay minutes.

<a id="data"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>2 &nbsp;Data</b></span></div>

The Bureau of Transportation Statistics (BTS), U.S. Department of Transportation, collects flight-level on-time performance records to monitor airline operations. We use the supplied 2024 combined Parquet file, with one observation per flight. We exclude cancelled and diverted flights and incomplete records, then randomly sample up to 20,000 eligible flights with seed 42 to keep the classroom diagnostics manageable. The output above reports actual month coverage and sample counts. The original download filters and exact monthly extraction procedure require group confirmation.

In [ ]:
# A quick first look at the shape and the variable types.
print('Observations (rows):', df.shape[0])
print('Variables  (cols):', df.shape[1])
print()
df.info()

| Variable | What it measures | Type | Units / coding |
|---|---|---|---|
| ArrDelayMinutes | Arrival delay, early arrivals set to zero (Y) | Numeric | Minutes |
| Distance / distance100 | Flight distance (X) | Numeric | Miles / hundreds of miles |
| Reporting_Airline | Reporting carrier | Categorical | Airline code |
| Month | Calendar month | Categorical | 1–12 |
| DayOfWeek / Friday | Day of week / Friday indicator | Binary derived | 1 = Friday |
| December | December indicator | Binary derived | 1 = December |
| Origin | Departure airport (descriptive) | Categorical | Airport code |
| Year, Cancelled, Diverted | Sample filters | Numeric/binary | Year; 0 or 1 |

In [ ]:
raw[cols].isna().sum()

<a id="descriptives"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>3 &nbsp;Descriptive statistics and correlations</b></span></div>

The following tables summarize the common analysis sample. The earlier proposal reported a distance-delay correlation near −0.01; the new sample may differ.

In [ ]:
myvars = ['ArrDelayMinutes','distance100','Friday','December']
df[myvars].describe().T

In [ ]:
# Correlation table among the numeric variables of interest.
df[myvars].corr(numeric_only=True).round(2)

<a id="graphs"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>4 &nbsp;Charts and plots</b></span></div>

The plots show the distribution of arrival delays, their relationship with distance, and differences across carriers. These are descriptive associations, not causal estimates.

In [ ]:
plt.hist(df.ArrDelayMinutes,bins=50)
plt.xlabel('Arrival delay (minutes)'); plt.ylabel('Count'); plt.show()

In [ ]:
plt.scatter(df.distance100,df.ArrDelayMinutes,s=5,alpha=.15)
plt.xlabel('Distance (100 miles)'); plt.ylabel('Arrival delay (minutes)'); plt.show()

In [ ]:
sns.boxplot(data=df,x='Reporting_Airline',y='ArrDelayMinutes',showfliers=False)
plt.xticks(rotation=90); plt.xlabel('Carrier'); plt.ylabel('Arrival delay (minutes)'); plt.show()

In [ ]:
# (d) Correlation heatmap among your variables of interest.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(df[myvars].corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='PuBuGn', vmin=-1, vmax=1, ax=ax)
plt.show()

Our original prediction was that longer flights would have more opportunities for disruptions. The previous proposal reported a weak negative raw correlation instead. The new regression tests whether the relationship remains weak conditional on controls.

<a id="methodology"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>5 &nbsp;Methodology / empirical model</b></span></div>

The main model is
$$Y_i=\beta_0+\beta_1 Distance100_i+\beta_2 Friday_i+\sum_m\gamma_m Month_{im}+\sum_a\delta_a Airline_{ia}+u_i.$$
The focal coefficient is $\beta_1$, minutes of delay per additional 100 miles. Carrier and month indicators account for persistent airline differences and seasonality; Friday captures weekly scheduling patterns. The original positive distance hypothesis is retained as a prior expectation, not a newly claimed pre-test prediction. Friday and month effects are ambiguous. Departure delay and attributed delay components are excluded to avoid mechanical relationships. Actual weather, congestion and aircraft maintenance remain unobserved; their effects on delay may be positive, but their correlations with distance are unknown, so the direction of omitted-variable bias is uncertain. Operational efficiency may reduce delay, but its correlation with distance is also unknown.

Following Week 9, we use explicit dummy variables, an intercept, and `sm.OLS`. The first sorted observed category is the reference for each indicator set.

In [ ]:
y = df['ArrDelayMinutes'].astype(float)
airline = pd.get_dummies(df.Reporting_Airline.astype(str), prefix='airline', drop_first=True, dtype=float)
month = pd.get_dummies(df.Month.astype(int), prefix='month', drop_first=True, dtype=float)
print('Reference airline:',sorted(df.Reporting_Airline.astype(str).unique())[0])
print('Reference month:',min(df.Month))
X_short = sm.add_constant(pd.concat([df[['distance100','Friday']],month],axis=1).astype(float))
X_long = sm.add_constant(pd.concat([df[['distance100','Friday']],month,airline],axis=1).astype(float))
m_short = sm.OLS(y,X_short).fit()
model = sm.OLS(y,X_long)
ols_classic = model.fit()
ols_classic.summary()

<a id="results"></a>
## 6. Results and robustness
### 6.1 Main coefficient and Week 9 short/long comparison
The short model omits carrier indicators; the long model adds them using identical observations. A coefficient change is evidence of sensitivity to the controls, not proof that all omitted-variable bias is removed.

In [ ]:
m_long = ols_classic
emp_bias = m_short.params['distance100'] - m_long.params['distance100']
print('Short minus long:',emp_bias)
print(m_short.summary().tables[1])
comparison = pd.DataFrame({'Distance coefficient':[m_short.params['distance100'],m_long.params['distance100']], 'R-squared':[m_short.rsquared,m_long.rsquared]},index=['Short model','Long model'])
display(comparison)

### 6.2 Week 12: classical and HC0 standard errors

In [ ]:
ols_hc0 = model.fit(cov_type="HC0")
print('=== Classical ==='); print(ols_classic.summary().tables[1])
print('=== Robust HC0 ==='); print(ols_hc0.summary().tables[1])
results = pd.DataFrame([{'Model':label,'Coefficient':fit.params['distance100'],'SE':fit.bse['distance100'],'p':fit.pvalues['distance100'],'N':int(fit.nobs),'R2':fit.rsquared} for label,fit in [('Short',m_short),('Main classical',ols_classic),('Main HC0',ols_hc0)]])
display(results)
for label,fit in [('Main classical',ols_classic),('Main HC0',ols_hc0)]:
    print(f"{label}: {fit.params['distance100']:.4f} minutes per 100 miles; p={fit.pvalues['distance100']:.4g}; significant at 5%: {fit.pvalues['distance100'] < .05}")

### 6.3 Residual diagnostics (Week 12)

In [ ]:
plt.scatter(ols_classic.fittedvalues,ols_classic.resid,s=5,alpha=.15)
plt.axhline(0,ls='--'); plt.xlabel('Fitted arrival delay (minutes)'); plt.ylabel('Residuals (minutes)'); plt.show()
bp_lm,bp_lm_p,_,_ = sm_diagnostic.het_breuschpagan(ols_classic.resid,X_long)
white_lm,white_lm_p,_,_ = sm_diagnostic.het_white(ols_classic.resid,X_long)
print({'BP_p':float(bp_lm_p),'White_p':float(white_lm_p)})
print('Reject homoskedasticity at 5%' if min(bp_lm_p,white_lm_p)<.05 else 'No strong evidence against homoskedasticity')

### 6.4 Interaction
December may change the distance-delay relationship because of seasonal demand and weather. If December is absent, use Friday instead. The main effect of the indicator is already included in the model.

In [ ]:
indicator = 'December' if df.December.nunique()==2 else 'Friday'
if df[indicator].nunique()==2:
    X_interaction = X_long.copy()
    term = 'distance100_x_' + indicator
    X_interaction[term] = df.distance100 * df[indicator]
    m_interaction = sm.OLS(y,X_interaction).fit(cov_type='HC0')
    print(m_interaction.summary().tables[1])
    print('Slope outside group:',m_interaction.params['distance100'])
    print('Slope inside group:',m_interaction.params['distance100']+m_interaction.params[term])
else:
    print('Interaction unavailable: no indicator variation.')

<a id="conclusion"></a>
## 7. Conclusion
The revised specification avoids using components of arrival delay as explanatory variables. After executing the notebook, the group should report the observed coefficient, its practical magnitude, whether significance changes with HC0, and whether adding carrier indicators changes the conclusion. HC0 addresses heteroskedasticity but not dependence among flights or omitted confounding. No causal interpretation is warranted. Numerical conclusions are pending execution.

<a id="references"></a>
<div style="border-left:7px solid #0C5A5C; background:#EFEAF0; padding:10px 20px; margin:14px 0 6px 0; border-radius:0 6px 6px 0;"><span style="font-size:19pt; color:#0C5A5C;"><b>8 &nbsp;References</b></span></div>

Sources carried over from the flight proposal (group must verify titles and findings before submission):
1. https://www.sciencedirect.com/science/article/pii/S0969699707000646
2. https://www.sciencedirect.com/science/article/pii/S0191261522000741
3. https://doi.org/10.1002/joc.70357

Data: BTS On-Time Performance, https://www.transtats.bts.gov/ (exact table and extraction filters to confirm).

<hr style="border-top:3px solid #0C5A5C; margin:18px 0;">

**Before you submit:** hide the code and read the document top to bottom &mdash; it should stand on its own as a research paper. Then export to HTML and submit **both** files.

```
# Run in a terminal, or in a cell with a leading ! , from the notebook folder:
jupyter nbconvert --to html YOUR_NOTEBOOK.ipynb
```

AI disclosure: We asked ChatGPT to adapt the final project template and implement the Week 9 and Week 12 methods using our supplied data and prior proposal; the group must run and verify the results and document its own subsequent work.